In [1]:
!git clone https://github.com/sharul-ayub/malaysia-bank-employee-sentiment-analysis.git
%cd malaysia-bank-employee-sentiment-analysis

Cloning into 'malaysia-bank-employee-sentiment-analysis'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 82 (delta 34), reused 27 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 436.55 KiB | 5.26 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/malaysia-bank-employee-sentiment-analysis


# 03 — Text Cleaning and Normalisation

This notebook starts from the manually checked binary-labelled sentence dataset and performs the cleaning steps from the original project code.

Main steps:

- lowercase
- duplicate removal
- whitespace cleanup
- encoding repair
- contraction expansion
- selected symbol handling
- number removal
- apostrophe/possessive cleanup
- bank/place masking
- slang, abbreviation and typo normalisation
- non-English/unrecognised-word inspection

> **Ordering adjustment:** contraction expansion is performed before apostrophe removal because the original notes explicitly state that contractions such as `don't` and `wasn't` should be expanded before punctuation cleanup.

In [2]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(".")
INPUT_PATH = PROJECT_ROOT / "data" / "labeled" / "03_sentence_labeled_raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_text_labelled = pd.read_csv(INPUT_PATH)

print("Loaded:", INPUT_PATH)
print("Rows:", len(df_text_labelled))
display(df_text_labelled.head())

Loaded: data/labeled/03_sentence_labeled_raw.csv
Rows: 1288


,sentiment,review_text
0,Negative,The environment is good but the basic is too low.
1,Negative,Still a lot of manual work and very volume bas...
2,Negative,Only give short break
3,Negative,but Depends on Your Tolerance with the Culture
4,Negative,Everything is manual and use a lot of paper.


## Install text-normalisation packages

In [3]:
%pip install -q ftfy contractions pyenchant
import ftfy
import contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00


## Lowercase and remove exact duplicate sentences

In [5]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .astype(str)
    .str.lower()
)

duplicate_count = df_text_labelled.duplicated(
    subset="review_text"
).sum()

print("Duplicate rows found:", duplicate_count)

before = len(df_text_labelled)

df_text_labelled = (
    df_text_labelled
    .drop_duplicates(subset="review_text", keep="first")
    .reset_index(drop=True)
)

print("Rows before:", before)
print("Rows after:", len(df_text_labelled))
print("Rows removed:", before - len(df_text_labelled))

Duplicate rows found: 15
Rows before: 1288
Rows after: 1273
Rows removed: 15


## Fix broken encoding and expand contractions

In [6]:
# Keep original text
df_text_labelled["review_text_original"] = df_text_labelled["review_text"]

# Step 1: Fix broken encoding
df_text_labelled["review_text_encoding_fixed"] = (
    df_text_labelled["review_text_original"]
    .apply(ftfy.fix_text)
)

# Step 2: Expand contractions
df_text_labelled["review_text_cleaned"] = (
    df_text_labelled["review_text_encoding_fixed"]
    .apply(contractions.fix)
)

# Identify rows changed by encoding fix
df_text_labelled["encoding_changed"] = (
    df_text_labelled["review_text_original"]
    != df_text_labelled["review_text_encoding_fixed"]
)

# Identify rows changed by contraction expansion
df_text_labelled["contraction_changed"] = (
    df_text_labelled["review_text_encoding_fixed"]
    != df_text_labelled["review_text_cleaned"]
)

# Show only rows where something changed
changed_rows = df_text_labelled[
    df_text_labelled["encoding_changed"]
    | df_text_labelled["contraction_changed"]
]

display(
    changed_rows[
        [
            "review_text_original",
            "review_text_encoding_fixed",
            "review_text_cleaned",
            "encoding_changed",
            "contraction_changed",
        ]
    ]
)

,review_text_original,review_text_encoding_fixed,review_text_cleaned,encoding_changed,contraction_changed
13,"don't push yourself to hard, but learn and try...","don't push yourself to hard, but learn and try...","do not push yourself to hard, but learn and tr...",False,True
14,even if it wasnât recognised by management.,even if it wasn't recognised by management.,even if it was not recognised by management.,True,True
20,but in terms of workloads there werenât muc...,but in terms of workloads there weren't much ...,but in terms of workloads there were not much...,True,True
27,if ur fresh graduate just try to gain experien...,if ur fresh graduate just try to gain experien...,if you are fresh graduate just try to gain exp...,False,True
39,"good pay and bonus, yet there's no work life b...","good pay and bonus, yet there's no work life b...","good pay and bonus, yet there is no work life ...",False,True
...,...,...,...,...,...
1235,"work like dog, work until you die, no one gonn...","work like dog, work until you die, no one gonn...","work like dog, work until you die, no one goin...",False,True
1241,please donât work here this company is reall...,please don't work here this company is really ...,please do not work here this company is really...,True,True
1242,toxic cultureâcolleagues fought over custome...,toxic culture—colleagues fought over customers...,toxic culture—colleagues fought over customers...,True,False
1267,the worst place and management i've ever exper...,the worst place and management i've ever exper...,the worst place and management i have ever exp...,False,True


## Normalise whitespace

In [7]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Preserve the meaning of `>` before symbol cleanup

The original project notes convert `>` to the phrase `more than` rather than deleting it.

In [8]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(">", " more than ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove irrelevant numbers

In [9]:
rows_with_numbers = df_text_labelled[
    df_text_labelled["review_text"].str.contains(r"\d", regex=True, na=False)
]

print("Rows containing numbers before cleaning:", len(rows_with_numbers))

df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

Rows containing numbers before cleaning: 43


## Remove selected punctuation and symbols

In [10]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace("-", " ", regex=False)
    .str.replace(r"[-–—]", " ", regex=True)
    .str.replace(r"\.{2,}", " ", regex=True)
    .str.replace("…", " ", regex=False)
    .str.replace(".", " ", regex=False)
    .str.replace(r'[&!\/,“”"()#@\?:+%]', " ", regex=True)
    .str.replace("\ufe0f", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove unnecessary apostrophes while preserving apostrophes between letters

In [11]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"(?<![a-zA-Z])'|'(?![a-zA-Z])", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Replace masked system name

In [12]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"r\*w\*\*", "system", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove possessive `'s`

In [13]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\b([a-zA-Z]+)'s\b", r"\1", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Manual mappings: banks, abbreviations, slang, places and typos

In [14]:
bank_terms = {
    "cimb": "bankname",
    "rhb": "bankname",
    "uob": "bankname",
    "citi": "bankname",
    "citibank": "bankname",
    "maybank": "bankname",
    "pbb": "bankname",
    "pb": "bankname",
    "hsbc": "bankname",
    "ocbc": "bankname",
    "bank islam": "bankname",
    "bank rakyat": "bankname",
    "public bank": "bankname",
    "public bank berhad": "bankname"
}

english_short_forms = {
    "u": "you",
    "k": "",
    "comm": "commission",
    "dept": "department",
    "mgmt": "management",
    "lvl": "level",
    "yr": "year",
    "yrs": "years",
    "mon": "monday",
    "fri": "friday",
    "eg": "example",
    "ie": "that is",
    "ur": "your",
    "ok": "okay",
    "tgter": "together",
    "hlp": "help",
    "eq": "emotional intelligence",
    "flexi": "flexible",
    "lol": "laughing out loud",
    "bos": "boss",
    "ocr": "optical character recognition",
    "ai": "artificial intelligence",
    "ull": "you will",
    "eventho": "even though",
    "que": "queue",
    "geas": "graduate employability enhancement scheme"
}

work_abbreviations = {
    "kpi": "key performance indicator",
    "wfh": "work from home",
    "wlb": "work life balance",
    "ot": "overtime",
    "ots": "overtime",
    "sop": "standard operating procedure",
    "sops": "standard operating procedures",
    "hrbp": "human resource business partner",
    "sr": "service request",
    "oic": "officer in charge",
    "oics": "officers in charge",
    "hq": "headquarters",
    "tl": "team leader",
    "tm": "team manager",
    "al": "annual leave",
    "hl": "housing loan",
    "hp": "car loan",
    "hra": "housing rent allowance",
    "da": "dearness allowance",
    "jd": "job description",
    "mc": "medical certificate",
    "sme": "small and medium sized enterprises",
    "iso": "international organization for standardization",
    "epf": "employee provident fund",
    "cia": "chief internal auditor",
    "concall": "conference call",
    "ees": "employee engagement surveys",
    "bau": "business as usual",
    "ceo": "chief executive officer",
    "rorg": "reorganization"
}

malay_slang = {
    "lah": "",
    "tak": "not",
    "x": "not",
    "pandai bodek": "suck up",
    "menara": "tower",
    "tai chi": "avoid responsibility",
    "buli": "bully"
}

place_terms = dict.fromkeys(
    [
        "alor setar",
        "alor",
        "setar",
        "ipoh",
        "kedah",
        "penang",
        "malaysia",
        "kinabalu"
    ],
    "placename"
)

typo_corrections = {
    "stayback": "stay back",
    "commision": "commission",
    "commisions": "commissions",
    "commisioning": "commissioning",
    "comissioning": "commissioning",
    "work‑life balance": "work life balance",
    "hierarch": "hierarchy",
    "preffered": "preferred",
    "chnage": "change",
    "alot": "a lot",
    "beed": "need",
    "imcludes": "includes",
    "diaturb": "disturbed",
    "teamates": "teammates",
    "increament": "increment",
    "profesional": "professional",
    "enda": "end",
    "implimented": "implemented",
    "enviroment": "environment",
    "reffaller": "referral",
    "collegues": "colleagues",
    "newcommer": "newcomer",
    "oppertunity": "opportunity",
    "everday": "everyday",
    "jobscope": "job scope",
    "decaded": "outdated",
    "overprocess": "overprocessed",
    "manangement": "management",
    "knowledges": "knowledge",
    "pleasent": "pleasant",
    "unproper": "improper",
    "contidence": "confidence",
    "internaltional": "international",
    "advices": "advice",
    "friendlt": "friendly",
    "gor": "for",
    "goodplace": "good place",
    "paperworks": "paperwork",
    "fundwork": "fun work",
    "experince": "experience",
    "worklife": "work life",
    "rwally": "really",
    "renumeration": "remuneration",
    "unhumanity": "inhumanity",
    "persomal": "personal"
}

word_map = {}
word_map.update(english_short_forms)
word_map.update(work_abbreviations)
word_map.update(malay_slang)
word_map.update(place_terms)
word_map.update(typo_corrections)

def manual_map_text(text):
    text = str(text).lower()

    phrase_map = {}
    phrase_map.update(bank_terms)
    phrase_map.update({
        k: v for k, v in word_map.items()
        if " " in k
    })

    for old, new in sorted(
        phrase_map.items(),
        key=lambda x: len(x[0]),
        reverse=True
    ):
        text = re.sub(
            r"\b" + re.escape(old) + r"\b",
            new,
            text
        )

    single_word_map = {
        k: v for k, v in word_map.items()
        if " " not in k
    }

    words = text.split()
    mapped_words = [
        single_word_map.get(word, word)
        for word in words
    ]

    return re.sub(
        r"\s+",
        " ",
        " ".join(mapped_words)
    ).strip()

df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .apply(manual_map_text)
)

## Inspect unrecognised English words

`pyenchant` may require the system Enchant library. In Google Colab run the apt command first.

In [ ]:
# For Google Colab only, uncomment if Enchant is not installed:
# !apt-get update -qq
# !apt-get install -y enchant-2

In [15]:
try:
    import enchant

    d = enchant.Dict("en_US")

    ignore_or_keep_words = {
        "recognised", "programmes", "sunday", "wednesday", "adulting",
        "benchmarking", "siloed", "learnt", "digitalize", "hospitalisation",
        "chinese", "reportings", "townhall", "upskilling", "christmas",
        "programme", "favouritism", "organisation", "roleplay", "hahaha",
        "travelling", "monday", "friday", "glassdoor", "chatgpt",
        "datalake", "softskills", "islamic", "trainings", "desking",
        "mentorship", "flexcare", "freshie", "elearning", "employability",
        "specialised", "centric", "centre", "gaslighting", "positivity",
        "kickstart", "frontend", "placename", "bankname"
    }

    ignore_words = ignore_or_keep_words.union(
        set(typo_corrections.keys())
    )

    def find_non_english_words(text):
        words = re.findall(
            r"\b[a-zA-Z]+\b",
            str(text).lower()
        )

        return sorted({
            word
            for word in words
            if word not in ignore_words
            and not d.check(word)
        })

    df_text_labelled["non_english_words"] = (
        df_text_labelled["review_text"]
        .apply(find_non_english_words)
    )

    rows_non_english = df_text_labelled[
        df_text_labelled["non_english_words"].str.len() > 0
    ]

    display(
        rows_non_english[
            ["review_text", "non_english_words"]
        ].reset_index()
    )

except Exception as e:
    print("Enchant inspection skipped:", e)
    df_text_labelled["non_english_words"] = [[] for _ in range(len(df_text_labelled))]

Enchant inspection skipped: The 'enchant' C library was not found and maybe needs to be installed.
See  https://pyenchant.github.io/pyenchant/install.html
for details



## Save cleaned and normalised sentences

In [16]:
output_path = PROCESSED_DIR / "02_sentences_labeled_processed.csv"

df_text_labelled.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)
print("Rows:", len(df_text_labelled))
display(df_text_labelled.head())

Saved: data/processed/02_sentences_labeled_processed.csv
Rows: 1273


,sentiment,review_text,review_text_original,review_text_encoding_fixed,review_text_cleaned,encoding_changed,contraction_changed,non_english_words
0,Negative,the environment is good but the basic is too low,the environment is good but the basic is too low.,the environment is good but the basic is too low.,the environment is good but the basic is too low.,False,False,[]
1,Negative,still a lot of manual work and very volume bas...,still a lot of manual work and very volume bas...,still a lot of manual work and very volume bas...,still a lot of manual work and very volume bas...,False,False,[]
2,Negative,only give short break,only give short break,only give short break,only give short break,False,False,[]
3,Negative,but depends on your tolerance with the culture,but depends on your tolerance with the culture,but depends on your tolerance with the culture,but depends on your tolerance with the culture,False,False,[]
4,Negative,everything is manual and use a lot of paper,everything is manual and use a lot of paper.,everything is manual and use a lot of paper.,everything is manual and use a lot of paper.,False,False,[]
